In [3]:
import pandas as pd

df = pd.read_csv("/content/featured_violations.csv")

print(df.shape)
df.head()

/tmp/ipykernel_570/3143699017.py:3: DtypeWarning: Columns (14,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/featured_violations.csv")


(221348, 26)


,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,modified_datetime,...,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp,hour,day_of_week,month,weekend,peak_hour
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,2023-11-28 04:48:04.582978+00,...,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00,0.0,Monday,11.0,False,False
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00:00,2023-11-24 23:00:24.115257+00,...,NaN,NaN,NaN,NaN,NaN,22.0,Friday,11.0,False,False
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00:00,2023-11-28 04:47:02.33776+00,...,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00,0.0,Monday,11.0,False,False
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00:00,2023-11-18 04:46:57.216868+00,...,NaN,FKN00GL0003,SCOOTER,approved,2023-11-18 23:35:12.428+00,6.0,Thursday,11.0,False,False
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00:00,2023-11-28 02:44:50.46737+00,...,NaN,FKN00GL0004,TANKER,approved,2023-11-30 03:11:32.796+00,4.0,Wednesday,11.0,False,False


In [ ]:
df["created_datetime"] = pd.to_datetime(
    df["created_datetime"],
    errors="coerce"
)

df["date"] = df["created_datetime"].dt.date

print(df[["created_datetime","date"]].head())

           created_datetime        date
0 2023-11-20 00:28:46+00:00  2023-11-20
1 2023-11-24 22:46:46+00:00  2023-11-24
2 2023-11-20 00:27:46+00:00  2023-11-20
3 2023-11-16 06:47:46+00:00  2023-11-16
4 2023-11-22 04:56:46+00:00  2023-11-22


In [ ]:
daily_counts = (
    df.groupby(["location","date"])
      .size()
      .reset_index(name="violation_count")
)

print(daily_counts.shape)
daily_counts.head()

(31045, 3)


,location,date,violation_count
0,"1, 7th C Cross Road, Maistripalaya, Kormangala...",2024-04-03,1
1,"100 Feet Road, BM Sri Circle, Indiranagar, Ben...",2024-02-06,1
2,"100 Feet Road, Babasaheb Colony, Kodihalli, Be...",2024-01-28,1
3,"100 Feet Road, Gopal Reddy Layout, Banaswadi, ...",2024-03-06,1
4,"100 Feet Road, Gopal Reddy Layout, Banaswadi, ...",2024-03-15,1


In [ ]:
print(daily_counts.shape)

(31045, 3)


In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.02,
    random_state=42
)

daily_counts["anomaly"] = model.fit_predict(
    daily_counts[["violation_count"]]
)

daily_counts.head()

,location,date,violation_count,anomaly
0,"1, 7th C Cross Road, Maistripalaya, Kormangala...",2024-04-03,1,1
1,"100 Feet Road, BM Sri Circle, Indiranagar, Ben...",2024-02-06,1,1
2,"100 Feet Road, Babasaheb Colony, Kodihalli, Be...",2024-01-28,1,1
3,"100 Feet Road, Gopal Reddy Layout, Banaswadi, ...",2024-03-06,1,1
4,"100 Feet Road, Gopal Reddy Layout, Banaswadi, ...",2024-03-15,1,1


In [ ]:
daily_counts["anomaly"].value_counts()

,count
anomaly,
1,30425
-1,620


In [ ]:
anomalies = daily_counts[
    daily_counts["anomaly"] == -1
]

print(anomalies.shape)

anomalies.head()

(620, 4)


,location,date,violation_count,anomaly
82,"10th Cross Road, Block 1, Rajaji Nagar, Bengal...",2023-11-29,38,-1
297,"11th Cross Road, Acharya Sri Ramanuja Circle, ...",2024-02-27,39,-1
964,"15th Cross Road, Malleshwara Extension, Malles...",2023-11-30,29,-1
973,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-03,26,-1
976,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-06,45,-1


In [ ]:
top_anomalies = anomalies.sort_values(
    by="violation_count",
    ascending=False
)

top_anomalies.head(20)

,location,date,violation_count,anomaly
22227,"New Horizon College Road, New Horizon College ...",2024-02-23,247,-1
11295,"Chord Road, Manuvana, Vijaya Nagar, Bengaluru,...",2024-03-12,156,-1
22241,"New Horizon College Road, New Horizon College ...",2024-03-15,154,-1
19142,"MBT Road, Devasandra Junction, KR Puram, Benga...",2024-04-07,112,-1
22175,"New Horizon College Road, Embassy Tech Village...",2024-02-22,110,-1
23979,"Outer Ring Road, Vajram Onyx, Devara Beesana H...",2024-02-22,106,-1
5528,"4th Main Road, Sri Basaveshwara HBCS, Phase 1,...",2024-02-15,101,-1
26958,"Shri Ramalingeshwara Cave Temple Road, Royal M...",2024-03-13,101,-1
29661,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",2024-02-21,100,-1
26134,"Sahakar Nagar Road, Fortuna Acacia, Byatarayan...",2024-02-10,92,-1


In [ ]:
anomalies.to_csv(
    "anomalies.csv",
    index=False
)

print("Saved")

Saved


In [ ]:
top_anomalies.head(20)

,location,date,violation_count,anomaly
22227,"New Horizon College Road, New Horizon College ...",2024-02-23,247,-1
11295,"Chord Road, Manuvana, Vijaya Nagar, Bengaluru,...",2024-03-12,156,-1
22241,"New Horizon College Road, New Horizon College ...",2024-03-15,154,-1
19142,"MBT Road, Devasandra Junction, KR Puram, Benga...",2024-04-07,112,-1
22175,"New Horizon College Road, Embassy Tech Village...",2024-02-22,110,-1
23979,"Outer Ring Road, Vajram Onyx, Devara Beesana H...",2024-02-22,106,-1
5528,"4th Main Road, Sri Basaveshwara HBCS, Phase 1,...",2024-02-15,101,-1
26958,"Shri Ramalingeshwara Cave Temple Road, Royal M...",2024-03-13,101,-1
29661,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",2024-02-21,100,-1
26134,"Sahakar Nagar Road, Fortuna Acacia, Byatarayan...",2024-02-10,92,-1


In [ ]:
daily_counts.to_csv(
    "daily_counts.csv",
    index=False
)



print("Files saved successfully")

Files saved successfully


In [ ]:
anomalies.describe()

,violation_count,anomaly
count,620.000000,620.0
mean,37.133871,-1.0
std,18.325009,0.0
min,23.000000,-1.0
25%,26.000000,-1.0
50%,32.000000,-1.0
75%,42.000000,-1.0
max,247.000000,-1.0


In [ ]:
anomalies["violation_count"].describe()

,violation_count
count,620.000000
mean,37.133871
std,18.325009
min,23.000000
25%,26.000000
50%,32.000000
75%,42.000000
max,247.000000


In [ ]:
anomalies["severity"] = anomalies["violation_count"].apply(
    lambda x: "HIGH" if x > 150
    else "MEDIUM" if x > 80
    else "LOW"
)

anomalies.head()

/tmp/ipykernel_4858/1272881300.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anomalies["severity"] = anomalies["violation_count"].apply(


,location,date,violation_count,anomaly,severity
82,"10th Cross Road, Block 1, Rajaji Nagar, Bengal...",2023-11-29,38,-1,LOW
297,"11th Cross Road, Acharya Sri Ramanuja Circle, ...",2024-02-27,39,-1,LOW
964,"15th Cross Road, Malleshwara Extension, Malles...",2023-11-30,29,-1,LOW
973,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-03,26,-1,LOW
976,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-06,45,-1,LOW


In [ ]:
anomalies["alert"] = (
    "Violation spike detected at "
    + anomalies["location"]
)

/tmp/ipykernel_4858/4064850975.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anomalies["alert"] = (


In [ ]:
anomalies.to_csv(
    "anomaly_alerts.csv",
    index=False
)

In [ ]:
anomalies["severity"] = anomalies["violation_count"].apply(
    lambda x: "CRITICAL" if x >= 200
    else "HIGH" if x >= 100
    else "MEDIUM"
)

anomalies.head()

/tmp/ipykernel_4858/5075514.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anomalies["severity"] = anomalies["violation_count"].apply(


,location,date,violation_count,anomaly,severity,alert
82,"10th Cross Road, Block 1, Rajaji Nagar, Bengal...",2023-11-29,38,-1,MEDIUM,"Violation spike detected at 10th Cross Road, B..."
297,"11th Cross Road, Acharya Sri Ramanuja Circle, ...",2024-02-27,39,-1,MEDIUM,"Violation spike detected at 11th Cross Road, A..."
964,"15th Cross Road, Malleshwara Extension, Malles...",2023-11-30,29,-1,MEDIUM,"Violation spike detected at 15th Cross Road, M..."
973,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-03,26,-1,MEDIUM,"Violation spike detected at 15th Cross Road, M..."
976,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-06,45,-1,MEDIUM,"Violation spike detected at 15th Cross Road, M..."


In [ ]:
anomalies["alert_message"] = (
    "Parking violation spike detected at "
    + anomalies["location"]
)

/tmp/ipykernel_4858/2873828718.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anomalies["alert_message"] = (


In [ ]:
anomalies = anomalies.reset_index(drop=True)

anomalies["alert_id"] = [
    f"ALT{i+1:04d}"
    for i in range(len(anomalies))
]

In [ ]:
alerts = anomalies[
    [
        "alert_id",
        "location",
        "date",
        "violation_count",
        "severity",
        "alert_message"
    ]
]

alerts.head()

,alert_id,location,date,violation_count,severity,alert_message
0,ALT0001,"10th Cross Road, Block 1, Rajaji Nagar, Bengal...",2023-11-29,38,MEDIUM,Parking violation spike detected at 10th Cross...
1,ALT0002,"11th Cross Road, Acharya Sri Ramanuja Circle, ...",2024-02-27,39,MEDIUM,Parking violation spike detected at 11th Cross...
2,ALT0003,"15th Cross Road, Malleshwara Extension, Malles...",2023-11-30,29,MEDIUM,Parking violation spike detected at 15th Cross...
3,ALT0004,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-03,26,MEDIUM,Parking violation spike detected at 15th Cross...
4,ALT0005,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-06,45,MEDIUM,Parking violation spike detected at 15th Cross...


In [ ]:
alerts.to_csv(
    "anomaly_alerts.csv",
    index=False
)

print("Alert file generated")

Alert file generated


In [ ]:
alerts.head(20)

,alert_id,location,date,violation_count,severity,alert_message
0,ALT0001,"10th Cross Road, Block 1, Rajaji Nagar, Bengal...",2023-11-29,38,MEDIUM,Parking violation spike detected at 10th Cross...
1,ALT0002,"11th Cross Road, Acharya Sri Ramanuja Circle, ...",2024-02-27,39,MEDIUM,Parking violation spike detected at 11th Cross...
2,ALT0003,"15th Cross Road, Malleshwara Extension, Malles...",2023-11-30,29,MEDIUM,Parking violation spike detected at 15th Cross...
3,ALT0004,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-03,26,MEDIUM,Parking violation spike detected at 15th Cross...
4,ALT0005,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-06,45,MEDIUM,Parking violation spike detected at 15th Cross...
5,ALT0006,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-17,30,MEDIUM,Parking violation spike detected at 15th Cross...
6,ALT0007,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-21,28,MEDIUM,Parking violation spike detected at 15th Cross...
7,ALT0008,"15th Cross Road, Malleshwara Extension, Malles...",2024-03-12,23,MEDIUM,Parking violation spike detected at 15th Cross...
8,ALT0009,"17th Cross Road, Sector 7, HSR Layout, Bengalu...",2024-02-11,24,MEDIUM,Parking violation spike detected at 17th Cross...
9,ALT0010,"17th Main Road, KPTCL Quarters, Block 2, Rajaj...",2024-04-03,23,MEDIUM,Parking violation spike detected at 17th Main ...


In [ ]:
alerts.head(10)

,alert_id,location,date,violation_count,severity,alert_message
0,ALT0001,"10th Cross Road, Block 1, Rajaji Nagar, Bengal...",2023-11-29,38,MEDIUM,Parking violation spike detected at 10th Cross...
1,ALT0002,"11th Cross Road, Acharya Sri Ramanuja Circle, ...",2024-02-27,39,MEDIUM,Parking violation spike detected at 11th Cross...
2,ALT0003,"15th Cross Road, Malleshwara Extension, Malles...",2023-11-30,29,MEDIUM,Parking violation spike detected at 15th Cross...
3,ALT0004,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-03,26,MEDIUM,Parking violation spike detected at 15th Cross...
4,ALT0005,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-06,45,MEDIUM,Parking violation spike detected at 15th Cross...
5,ALT0006,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-17,30,MEDIUM,Parking violation spike detected at 15th Cross...
6,ALT0007,"15th Cross Road, Malleshwara Extension, Malles...",2024-02-21,28,MEDIUM,Parking violation spike detected at 15th Cross...
7,ALT0008,"15th Cross Road, Malleshwara Extension, Malles...",2024-03-12,23,MEDIUM,Parking violation spike detected at 15th Cross...
8,ALT0009,"17th Cross Road, Sector 7, HSR Layout, Bengalu...",2024-02-11,24,MEDIUM,Parking violation spike detected at 17th Cross...
9,ALT0010,"17th Main Road, KPTCL Quarters, Block 2, Rajaj...",2024-04-03,23,MEDIUM,Parking violation spike detected at 17th Main ...


In [ ]:
alerts.sort_values(
    by="violation_count",
    ascending=False
).head(20)

,alert_id,location,date,violation_count,severity,alert_message
414,ALT0415,"New Horizon College Road, New Horizon College ...",2024-02-23,247,CRITICAL,Parking violation spike detected at New Horizo...
175,ALT0176,"Chord Road, Manuvana, Vijaya Nagar, Bengaluru,...",2024-03-12,156,HIGH,Parking violation spike detected at Chord Road...
422,ALT0423,"New Horizon College Road, New Horizon College ...",2024-03-15,154,HIGH,Parking violation spike detected at New Horizo...
320,ALT0321,"MBT Road, Devasandra Junction, KR Puram, Benga...",2024-04-07,112,HIGH,"Parking violation spike detected at MBT Road, ..."
391,ALT0392,"New Horizon College Road, Embassy Tech Village...",2024-02-22,110,HIGH,Parking violation spike detected at New Horizo...
441,ALT0442,"Outer Ring Road, Vajram Onyx, Devara Beesana H...",2024-02-22,106,HIGH,Parking violation spike detected at Outer Ring...
68,ALT0069,"4th Main Road, Sri Basaveshwara HBCS, Phase 1,...",2024-02-15,101,HIGH,Parking violation spike detected at 4th Main R...
520,ALT0521,"Shri Ramalingeshwara Cave Temple Road, Royal M...",2024-03-13,101,HIGH,Parking violation spike detected at Shri Ramal...
573,ALT0574,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",2024-02-21,100,HIGH,Parking violation spike detected at Unnamed Ro...
477,ALT0478,"Sahakar Nagar Road, Fortuna Acacia, Byatarayan...",2024-02-10,92,MEDIUM,Parking violation spike detected at Sahakar Na...


In [ ]:
daily_counts.to_csv(
    "daily_counts.csv",
    index=False
)

anomalies.to_csv(
    "anomalies.csv",
    index=False
)

alerts.to_csv(
    "anomaly_alerts.csv",
    index=False
)

In [ ]:
from google.colab import files

files.download("daily_counts.csv")
files.download("anomalies.csv")
files.download("anomaly_alerts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>